# BERTopic Training — 2x T4 GPU (Kaggle)

**Project:** DNLPA — Vietnamese Tech Trend Radar  
**Owner:** Member 3 — ML Engineer  
**Output chuẩn theo:** `docs/data_flow_schema_evolution.md`

### File cần upload lên Kaggle Dataset (`dnlpa-training-data`):

```
dnlpa-training-data/
├── stg_posts_core.csv          ← dữ liệu chính (164K rows)
├── slang_dict.json             ← từ data/slang_dict.json
├── slang_normalizer.py         ← từ preprocessing/slang_normalizer.py
└── stopwords_vi.txt            ← từ data/stopwords_vi.txt
```

> `slang_dict.json` + `slang_normalizer.py` giúp chuẩn hóa teencode VOZ trước khi encode  
> (ví dụ: "ko dc" → "không được", "thik" → "thích")  
> Notebook tự xử lý và lưu kết quả ra `/kaggle/working/`

## Cell 1 — Kiểm tra GPU

In [1]:
import subprocess, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {name} — {mem:.1f} GB VRAM")

assert torch.cuda.device_count() >= 1, "Cần ít nhất 1 GPU — bật GPU Accelerator trong Settings"

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 — 14.6 GB VRAM
  GPU 1: Tesla T4 — 14.6 GB VRAM


## Cell 2 — Cài packages

In [ ]:
%%capture
# hdbscan>=0.8.38 có numpy 2.x compatible wheels (0.8.33 chỉ hỗ trợ numpy 1.x)
!pip install bertopic==0.16.4 \
             sentence-transformers==2.7.0 \
             umap-learn==0.5.6 \
             hdbscan>=0.8.38 \
             gensim==4.3.2 \
             pyarrow==14.0.0 \
             --quiet

## Cell 3 — Import & Config

In [ ]:
import json, os, re, time, pickle, sys, unicodedata, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

import torch
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

warnings.filterwarnings('ignore')

# ── Đường dẫn input (file upload lên Kaggle dataset) ──
_DATASET_DIR  = "/kaggle/input/datasets/anhqunhong/data-training-bertopic"
RAW_CSV           = f"{_DATASET_DIR}/stg_posts_core.csv"
SLANG_DICT_PATH   = f"{_DATASET_DIR}/slang_dict.json"
SLANG_MODULE_PATH = _DATASET_DIR          # thư mục chứa slang_normalizer.py (sys.path cần dir)
STOPWORDS_PATH    = f"{_DATASET_DIR}/stopwords_vi.txt"

# ── Đường dẫn output (tất cả ra /kaggle/working/) ──
OUTPUT_DIR    = Path("/kaggle/working/bertopic_output")
PROCESSED_CSV = Path("/kaggle/working/training_data.csv")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Model config ──
EMBEDDING_MODEL = "vinai/phobert-base"   # đổi thành "BAAI/bge-m3" nếu muốn BGE
MODEL_VERSION   = "bertopic_v1"

# ── BERTopic hyperparams ──
N_NEIGHBORS      = 15
N_COMPONENTS     = 5
MIN_DIST         = 0.0
MIN_CLUSTER_SIZE = 15
MIN_SAMPLES      = 10
TOP_N_WORDS      = 10
BATCH_SIZE       = 256

# ── Giới hạn text ──
MAX_TEXT_LEN = 500   # PhoBERT max ~256 tokens ≈ 400-500 ký tự Vietnamese
MIN_TEXT_LEN = 10

print("Config loaded.")
print(f"Dataset dir: {_DATASET_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

## Cell 4 — Chuẩn bị Data từ `stg_posts_core.csv`

**Pipeline xử lý:**
1. Load `slang_normalizer.py` từ dataset → normalize teencode VOZ
2. Inline regex cleaning: HTML / URL / emoji / mention / forum-quote
3. Ghép `title (cleaned)` + `segmented_text` (đã word-segment sẵn)
4. Filter (len≥10, dedup post_id, dedup nội dung)
5. Truncate 500 ký tự → lưu `training_data.csv` ra `/kaggle/working/`

In [ ]:
# ══════════════════════════════════════════════════
# [A] Load SlangNormalizer từ preprocessing/slang_normalizer.py
# ══════════════════════════════════════════════════
sys.path.insert(0, SLANG_MODULE_PATH)
try:
    from slang_normalizer import SlangNormalizer
    slang_normalizer = SlangNormalizer(dict_path=SLANG_DICT_PATH)
    print(f"✅ SlangNormalizer loaded — {len(slang_normalizer.slang_dict):,} entries")
except (ImportError, FileNotFoundError) as e:
    print(f"⚠️  SlangNormalizer không load được ({e}) — bỏ qua normalize")
    class _NoopNorm:
        def normalize(self, text): return text
    slang_normalizer = _NoopNorm()

# ══════════════════════════════════════════════════
# [B] Inline cleaning utils (từ preprocessing/text_cleaner.py)
#     Không import TextPreprocessor vì nó phụ thuộc VnCoreNLP JAR (Java)
#     stg_posts_core.csv đã có segmented_text sẵn → không cần tokenize lại
# ══════════════════════════════════════════════════
_re_html    = re.compile(r"<[^>]+>")
_re_url     = re.compile(r"https?://\S+|www\.\S+")
_re_email   = re.compile(r"\S+@\S+\.\S+")
_re_mention = re.compile(r"@\w+")
_re_forum_q = re.compile(r"\b\w+\s+said.*?click to expand\s*",
                          flags=re.IGNORECASE | re.DOTALL)
_re_emoji   = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0\U000024C2-\U0001F251]+",
    flags=re.UNICODE,
)
_re_ws = re.compile(r"\s+")


def _clean_title(text: str) -> str:
    """
    Clean title trước khi ghép với segmented_text.
    Bước: forum-quote → HTML → URL → email → mention → emoji
          → NFC normalize → slang normalize → collapse whitespace
    """
    if not text or not isinstance(text, str):
        return ""
    text = _re_forum_q.sub(" ", text)
    text = _re_html.sub(" ", text)
    text = _re_url.sub(" ", text)
    text = _re_email.sub(" ", text)
    text = _re_mention.sub(" ", text)
    text = _re_emoji.sub(" ", text)
    text = unicodedata.normalize("NFC", text).lower()
    text = slang_normalizer.normalize(text)
    return _re_ws.sub(" ", text).strip()


# ══════════════════════════════════════════════════
# [C] Load raw CSV
# ══════════════════════════════════════════════════
print(f"\nLoading {RAW_CSV} ...")
raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f"Raw shape: {raw.shape}")
print(f"Columns  : {raw.columns.tolist()}")
print(f"Sources  : {raw['source'].value_counts().to_dict()}")

In [ ]:
df = raw.copy()

# ── [1] Build clean_text ──
# title: clean đầy đủ (HTML, URL, emoji, slang)
# segmented_text: đã word-segment sẵn bởi VnCoreNLP → dùng thẳng
# body: fallback khi không có segmented_text
def build_text(row) -> str:
    title = _clean_title(str(row.get('title', '') or ''))
    seg   = str(row.get('segmented_text', '') or '').strip()
    body  = str(row.get('body', '') or '').strip()
    text  = seg if seg else body
    if title and title.lower() not in text.lower():
        text = f"{title} {text}"
    return text.strip()

df['clean_text'] = df.apply(build_text, axis=1)

# ── [2] Filter ──
before = len(df)
df = df[df['clean_text'].str.len() >= MIN_TEXT_LEN]
df = df.dropna(subset=['clean_text'])
df = df.drop_duplicates(subset=['post_id'])
df = df.drop_duplicates(subset=['clean_text'])
print(f"After filter: {len(df):,} rows (removed {before - len(df):,})")

# ── [3] Truncate (PhoBERT max ~256 tokens ≈ 400-500 ký tự) ──
df['clean_text'] = df['clean_text'].str[:MAX_TEXT_LEN]

# ── [4] Giữ cột cần thiết ──
out = df[['post_id', 'clean_text', 'source']].copy()
out['post_id'] = out['post_id'].astype(str)

print(f"\nFinal training data : {len(out):,} rows")
print(f"Source breakdown    : {out['source'].value_counts().to_dict()}")
print(f"Text length (mean)  : {out['clean_text'].str.len().mean():.0f} chars")

# Kiểm tra mẫu sau khi clean
print("\nSample sau khi clean:")
for _, row in out.sample(3, random_state=42).iterrows():
    print(f"  [{row['source']}] {row['clean_text'][:100]}...")

# ── [5] Lưu ra /kaggle/working/ ──
out.to_csv(PROCESSED_CSV, index=False, encoding='utf-8')
print(f"\n✅ Saved → {PROCESSED_CSV}")
out.head(3)

## Cell 5 — Load & Validate Data đã xử lý

In [ ]:
df = pd.read_csv(PROCESSED_CSV)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

if len(df) < 5000:
    print(f"⚠️  Chỉ có {len(df)} docs — BERTopic hoạt động tốt nhất với ≥5K")

documents = df['clean_text'].tolist()
post_ids  = df['post_id'].astype(str).tolist()

print(f"\nSample:")
print(f"  {documents[0][:120]}...")

## Cell 6 — Encode với 2x T4 GPU (multi-process)

**Tại sao encode riêng trước khi fit BERTopic?**
- Encode là bước nặng nhất (~80% thời gian)
- Tách ra để dùng cả 2 GPU song song
- Encode 1 lần, có thể chạy lại UMAP/HDBSCAN nhiều lần mà không cần encode lại

In [ ]:
print(f"Loading {EMBEDDING_MODEL}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

n_gpus = torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

t0 = time.time()

if n_gpus >= 2:
    print("Encoding với 2x GPU (multi-process)...")
    pool = embedding_model.start_multi_process_pool(
        target_devices=['cuda:0', 'cuda:1']
    )
    embeddings = embedding_model.encode_multi_process(
        documents,
        pool=pool,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
    )
    embedding_model.stop_multi_process_pool(pool)

elif n_gpus == 1:
    print("Encoding với 1x GPU...")
    embeddings = embedding_model.encode(
        documents,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        device='cuda:0',
    )
else:
    print("⚠️  Không có GPU — encoding trên CPU (chậm)")
    embeddings = embedding_model.encode(
        documents,
        batch_size=32,
        show_progress_bar=True,
    )

elapsed = time.time() - t0
print(f"\nEncoding xong: {len(documents):,} docs trong {elapsed:.1f}s")
print(f"Embeddings shape: {embeddings.shape}")

np.save(OUTPUT_DIR / "embeddings.npy", embeddings)
print(f"Saved → {OUTPUT_DIR}/embeddings.npy")

In [ ]:
# Load lại nếu đã có (skip encode khi restart kernel)
embeddings_path = OUTPUT_DIR / "embeddings.npy"
if embeddings_path.exists():
    embeddings = np.load(embeddings_path)
    print(f"Loaded embeddings từ cache: {embeddings.shape}")

## Cell 7 — Build & Train BERTopic

In [ ]:
umap_model = UMAP(
    n_neighbors=N_NEIGHBORS,
    n_components=N_COMPONENTS,
    min_dist=MIN_DIST,
    metric='cosine',
    random_state=42,
    low_memory=False,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
    core_dist_n_jobs=-1,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=TOP_N_WORDS,
    calculate_probabilities=True,
    verbose=True,
)

print("Training BERTopic...")
t0 = time.time()
topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)
elapsed = time.time() - t0

n_topics   = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(t == -1 for t in topics)

print(f"\n{'='*50}")
print(f"TRAINING XONG trong {elapsed:.1f}s")
print(f"Topics tìm được : {n_topics}")
print(f"Outliers (-1)   : {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")
print(f"Docs có topic   : {len(topics) - n_outliers}")
print(f"{'='*50}")

## Cell 8 — Xem Topics

In [ ]:
topic_info = topic_model.get_topic_info()
print(f"Topic info ({len(topic_info)} rows):")
display(topic_info.head(15))

In [ ]:
print("Top words per topic:")
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = topic_model.get_topic(tid)
    top3  = [w for w, _ in words[:3]]
    count = topics.count(tid)
    print(f"  Topic {tid:2d} ({count:5d} docs): {' | '.join(top3)}")

## Cell 9 — Đánh giá Coherence

In [ ]:
print("Calculating Coherence C_V...")

topic_words = []
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)]
    topic_words.append(words)

texts      = [doc.split() for doc in documents]
dictionary = Dictionary(texts)

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v',
    processes=1,
)
coherence_cv = coherence_model.get_coherence()

print(f"\nCoherence C_V : {coherence_cv:.4f}")
print(f"Baseline LDA  : 0.5736")

if coherence_cv > 0.5:
    print("✅ Tốt")
elif coherence_cv > 0.35:
    print("⚠️  Chấp nhận được — thử tune hyperparams")
else:
    print("❌ Kém — tăng MIN_CLUSTER_SIZE hoặc N_NEIGHBORS")

## Cell 10 — Export theo spec schema (`stg_post_topics` + `stg_topics`)

In [ ]:
now = datetime.now(tz=timezone.utc)

# ── post_topic_assignment.parquet → stg_post_topics ──
if probs is not None and hasattr(probs, 'shape') and len(probs.shape) == 2:
    topic_probs = probs.max(axis=1).tolist()
else:
    topic_probs = [1.0] * len(topics)

post_topics_df = pd.DataFrame({
    'post_id'          : post_ids,
    'topic_id'         : [int(t) for t in topics],
    'topic_probability': [float(p) for p in topic_probs],
    'model_type'       : 'bertopic',
    'predicted_at'     : now,
})
post_topics_df.to_parquet(OUTPUT_DIR / 'post_topic_assignment.parquet', index=False)
print(f"✅ post_topic_assignment.parquet — {len(post_topics_df):,} rows")

# ── topics.parquet → stg_topics ──
topics_rows = []
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)]
    topics_rows.append({
        'topic_id'       : int(tid),
        'label'          : '_'.join(words[:3]),
        'top_keywords'   : words,
        'coherence_score': float(coherence_cv),
        'model_version'  : MODEL_VERSION,
        'created_at'     : now,
    })
topics_df = pd.DataFrame(topics_rows)
topics_df.to_parquet(OUTPUT_DIR / 'topics.parquet', index=False)
print(f"✅ topics.parquet — {len(topics_df)} topics")

display(topics_df[['topic_id', 'label', 'coherence_score', 'model_version']])

## Cell 11 — Lưu Model Artifacts

In [ ]:
model_dir = OUTPUT_DIR / 'bertopic_model'
model_dir.mkdir(exist_ok=True)

topic_model.save(
    str(model_dir / 'bertopic_model'),
    serialization='pickle',
    save_embedding_model=False,
)
print(f"✅ BERTopic model → {model_dir}/bertopic_model")

config = {
    'embedding_model' : EMBEDDING_MODEL,
    'model_version'   : MODEL_VERSION,
    'n_neighbors'     : N_NEIGHBORS,
    'n_components'    : N_COMPONENTS,
    'min_dist'        : MIN_DIST,
    'min_cluster_size': MIN_CLUSTER_SIZE,
    'min_samples'     : MIN_SAMPLES,
    'top_n_words'     : TOP_N_WORDS,
    'trained_on'      : now.isoformat(),
    'n_documents'     : len(documents),
    'n_topics'        : n_topics,
    'coherence_cv'    : float(coherence_cv),
}
with open(model_dir / 'config.pkl', 'wb') as f:
    pickle.dump(config, f)
print(f"✅ Config → {model_dir}/config.pkl")

with open(model_dir / 'topics.pkl', 'wb') as f:
    pickle.dump({'topics': topics, 'probs': probs}, f)
print(f"✅ Topics/probs → {model_dir}/topics.pkl")

# ── Tóm tắt toàn bộ output ──
print(f"\n{'='*55}")
print("OUTPUT /kaggle/working/ (tải về máy):")
print(f"{'='*55}")
all_files = sorted(Path('/kaggle/working').rglob('*'))
for f in all_files:
    if f.is_file():
        size = f.stat().st_size / 1024
        print(f"  {str(f.relative_to('/kaggle/working')):50s} {size:8.1f} KB")

print(f"\n{'='*55}")
print("SUMMARY")
print(f"{'='*55}")
print(f"  Documents trained : {len(documents):,}")
print(f"  Topics found      : {n_topics}")
print(f"  Outliers          : {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")
print(f"  Coherence C_V     : {coherence_cv:.4f}")
print(f"  Model version     : {MODEL_VERSION}")
print(f"  Embedding model   : {EMBEDDING_MODEL}")

## Cell 12 — Sau khi train: đẩy về local / HDFS

Tải toàn bộ `/kaggle/working/` về máy, sau đó:

```bash
# Copy model vào project local để test
cp -r bertopic_output/bertopic_model/ \
    output/task3.1_bertopic/output/bertopic_model/

# Đẩy lên HDFS (khi deploy cluster)
hdfs dfs -mkdir -p /data/models/bertopic/
hdfs dfs -put -f bertopic_output/bertopic_model/ /data/models/bertopic/
hdfs dfs -put -f bertopic_output/post_topic_assignment.parquet \
    /data/silver/post_topics/model=bertopic/
hdfs dfs -put -f bertopic_output/topics.parquet /data/silver/topics/
```